# Genshin Guide Route Generator (Colab)
Processes a Genshin Impact YouTube video into a localized, optimized route.

**Pipeline:** Download → Extract Minimaps + OCR → AKAZE Localization → Viterbi Optimization → Path Fusion → Export

> **Note:** Uses default calibration profile. For best results, create a calibration profile locally first via `python main.py` → Option 2, then paste the JSON into Cell 2.

In [ ]:
#@title 1. Setup
#@markdown Clone repo and install all dependencies.
!git clone https://github.com/Legend-Recalls/Genshin-Routes.git /content/Genshin-Routes
%cd /content/Genshin-Routes
!pip install -q opencv-python-headless yt-dlp easyocr numpy scipy tqdm Pillow scikit-image
!pip install -q --upgrade yt-dlp
# Install deno (JS runtime needed by yt-dlp for YouTube)
!curl -fsSL https://deno.land/install.sh | sh 2>/dev/null
import os
os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']
# Pre-download EasyOCR English model
import easyocr
easyocr.Reader(['en'], gpu=True)
print('Setup complete!')

In [ ]:
#@title 2. Input
#@markdown Enter the YouTube URL. Leave video_name blank to auto-detect.
YOUTUBE_URL = ""  #@param {type:"string"}
VIDEO_NAME = ""   #@param {type:"string"}

if not YOUTUBE_URL.strip():
    raise ValueError("Please enter a YouTube URL")
YOUTUBE_URL = YOUTUBE_URL.strip()
VIDEO_NAME = VIDEO_NAME.strip() if VIDEO_NAME.strip() else None
print(f'URL: {YOUTUBE_URL}')

In [ ]:
#@title 3. Download Video
%cd /content/Genshin-Routes

import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.downloader import download_video

print("Downloading video...")
t0 = time.time()
video_path = download_video(YOUTUBE_URL)
print(f"Downloaded: {video_path} ({time.time()-t0:.1f}s)")

if not VIDEO_NAME:
    VIDEO_NAME = "".join(c if c.isalnum() or c in "-_" else "_" for c in video_path.stem)[:80]
print(f"Video name: {VIDEO_NAME}")

In [ ]:
#@title 4. Extract Minimaps + OCR
#@markdown Extracts minimap frames, runs OCR on title crops, creates route.json.
%cd /content/Genshin-Routes

import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.extractor import extract_minimaps
from src.profile_manager import get_default_profile
from config import OUTPUT_DIR

t0 = time.time()
profile = get_default_profile()
print(f"Using default profile")

route_path = extract_minimaps(
    video_path=video_path,
    video_name=VIDEO_NAME,
    profile=profile,
    fps=3,
    output_base=OUTPUT_DIR,
)

import json
with open(route_path) as f:
    entries = json.load(f)
print(f"\nExtracted {len(entries)} frames ({time.time()-t0:.1f}s)")
print(f"Route: {route_path}")

# Show OCR results
captions = [e.get('caption_raw', '') for e in entries if e.get('caption_raw')]
if captions:
    print(f"Captions found: {len(captions)}")
    for c in set(captions):
        print(f"  - {c}")

In [ ]:
#@title 5. Localize Route
#@markdown AKAZE feature matching against map tiles to get world coordinates.
%cd /content/Genshin-Routes

import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.localizer import Localizer
from src.map_dataset import MapDataset
from config import OUTPUT_DIR, MAPS_DIR

map_dir = MAPS_DIR / "surface"
if not map_dir.exists():
    raise FileNotFoundError(
        f"Map tiles not found at {map_dir}. "
        "You need the maps/ directory with tile images for localization. "
        "Run the download script in reverse appsample/ first."
    )

print("Loading map tiles...")
dataset = MapDataset.load(map_dir)
print(f"Map loaded: {len(dataset.tiles)} tiles")

route_dir = OUTPUT_DIR / VIDEO_NAME
if not (route_dir / "route.json").exists():
    raise FileNotFoundError(f"No route.json in {route_dir}")

print("\nLocalizing...")
t0 = time.time()
localizer = Localizer(dataset)
localizer.localize_route(route_dir)
elapsed = time.time() - t0

import json
with open(route_dir / "route.json") as f:
    entries = json.load(f)
located = sum(1 for e in entries if e.get("lat") is not None)
print(f"\nLocalized {located}/{len(entries)} frames in {elapsed:.1f}s")

In [ ]:
#@title 6. Optimize + Fuse Path + Export
#@markdown Viterbi optimization, path smoothing, and export to JSON/GeoJSON/CSV.
%cd /content/Genshin-Routes

import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from config import OUTPUT_DIR
route_dir = OUTPUT_DIR / VIDEO_NAME
t0 = time.time()

# Optimize
print("Optimizing route...")
from src.optimizer import RouteOptimizer
optimizer = RouteOptimizer()
optimizer.optimize_route(route_dir)
print("  Done.")

# Path fusion
print("\nBuilding fused path...")
from src.path_fusion import build_path_file
try:
    build_path_file(route_dir)
    print("  Done.")
except Exception as e:
    print(f"  Skipped: {e}")

# Export
print("\nExporting...")
from src.exporter import RouteExporter
exporter = RouteExporter(route_dir)
exports_dir = route_dir / "exports"
exports_dir.mkdir(exist_ok=True)

exporter.to_json(exports_dir / "route.json")
print("  route.json")
exporter.to_geojson(exports_dir / "route.geojson")
print("  route.geojson")
exporter.to_csv(exports_dir / "route.csv")
print("  route.csv")

print(f"\nDone! ({time.time()-t0:.1f}s)")
print(f"Output: {route_dir}")

In [ ]:
#@title 7. Download Output
#@markdown Zips the output folder and downloads it.
%cd /content/Genshin-Routes

import shutil, os
from pathlib import Path
from google.colab import files

route_dir = Path("output") / VIDEO_NAME
if not route_dir.exists():
    raise FileNotFoundError(f"Output dir not found: {route_dir}")

zip_name = f"{VIDEO_NAME}.zip"
zip_path = f"/content/{zip_name}"
shutil.make_archive(f"/content/{VIDEO_NAME}", 'zip', "output", VIDEO_NAME)
zip_size = os.path.getsize(zip_path)
print(f"{zip_name}: {zip_size / 1024 / 1024:.1f} MB")

files.download(zip_path)
print("\nDownload started!")
print("Unzip into route_explorer/output/ to view in the map viewer.")